# Tạo ảnh host AI ngồi trong cảnh shop (ADR-036, ADR-060)

Notebook chạy **trên Kaggle**, không chạy trên laptop. Nó bật ComfyUI không giao diện web và tạo vài ảnh dọc 9:16: một nhân vật AI
(người không có thật) ngồi sau bàn, sau lưng là kệ hàng. Ảnh được chọn sẽ làm `start.png` cho InfiniteTalk
(`make_infinitetalk_video.ipynb`) để tạo video huấn luyện Ultralight.

**Quy tắc (CLAUDE.md, spec 005 FR-004, FR-005):**
- Gương mặt là người không có thật, trông là người trưởng thành, không giống người nổi tiếng hay người quen. Không đưa ảnh người
  thật vào notebook, kể cả ảnh tham khảo.
- Không vẽ logo hay thương hiệu có thật. Logo và ảnh sản phẩm của khách ghép sau bằng cảnh 009.
- Mỗi nhân vật chỉ dùng cho một khách, một shop.
- Ảnh và `provenance.json` **không đưa vào git**. Tải về laptop, lưu ở `assets/avatars/<CHARACTER_ID>/`.

**Cách chạy:** Accelerator = GPU T4, Internet = On, rồi Run All (hoặc Save & Run All). Kết quả ở `avatar_out/<CHARACTER_ID>/scenes/`,
kèm ảnh ghép `contact_sheet.jpg` để chọn nhanh.

Model: **FLUX.1-schnell** (Apache-2.0, ADR-036 đã cho phép), bản fp8 gộp sẵn của Comfy-Org, 4 bước. Không dùng FLUX.1-dev
(phi thương mại).

## 1. Cấu hình

Chỉ sửa ô này. Mọi giá trị được ghi vào `provenance.json`.

In [ ]:
CHARACTER_ID = "demo_host_02"  # one character = one customer, one shop
CREATED_BY = "jintran0501"  # who runs this notebook; required in provenance

# Shared description of the fictional host and the shop; a framing variant is appended per image.
# The shop is generic skincare (the demo products are a fictional sunscreen brand) and has no readable text.
SCENE_PROMPT = (
    "vertical photo of a livestream sales host in a bright modern skincare shop. A friendly fictional Vietnamese woman "
    "in her mid twenties, natural makeup, dark hair tied in a neat low ponytail, warm gentle smile with lips closed, "
    "looking straight into the camera, wearing a plain navy blue polo shirt with thin orange trim on the collar and "
    "sleeves, no logo. She sits behind a light natural wood desk, both hands resting on the desk with fingers loosely "
    "clasped together. Behind her: a softly lit blank white wall panel in the center, white display shelves on both "
    "sides with warm LED strip lighting, rows of plain white and pastel skincare bottles, tubes and boxes with no "
    "labels, a few small potted plants, black ceiling track spotlights. Even soft front lighting, shot on a phone "
    "camera, sharp focus on the face, realistic skin texture, natural colors, no text, no watermark"
)
FRAMINGS = {
    # As in the reference stream: more of the shop is visible, the face is small.
    "waist_up": "medium shot from the waist up, the desk fills the lower fifth of the frame, centered",
    # Closer: a larger face gives the lip-sync model more mouth pixels at 480p.
    "chest_up": "medium close-up from the chest up, desk edge and clasped hands at the bottom, centered",
}
RUN_FRAMINGS = ["chest_up"]  # run 1 used both framings
# Run 1 (2026-11-17) used [101, 202, 303, 404] on both framings. Run 2 only re-creates the chosen
# chest_up seed 202 (same prompt, seed and model revision) so its skin can be refined.
SEEDS = [202]
IMAGE_SIZE = (768, 1344)  # about 1 megapixel, 9:16 portrait; multiples of 16 for FLUX
STEPS = 4  # FLUX.1-schnell is distilled for 1-4 steps with cfg 1

# FLUX.1-schnell renders waxy, airbrushed skin ("plastic" in the user review of 2026-11-17). Two fixes are tried:
# 1. Refine: upscale the chosen image and re-sample it at low denoise with a skin-texture prompt. The face and the
#    layout stay; only fine detail (pores, fine hair, matte skin) is re-drawn.
REFINE_SOURCES = ["scene_chest_up_seed202.png"]
REFINE_SCALE = 1.5  # more pixels on the face gives the sampler room to draw pores
REFINE_DENOISE = [0.25, 0.35, 0.45]  # higher = more texture, but the face drifts further from the original
SKIN_PROMPT = (
    "unretouched candid smartphone photo, real human skin texture with visible pores on the nose and cheeks, fine "
    "peach fuzz, slight natural unevenness and tiny blemishes, matte skin with no shine, minimal makeup, natural lip "
    "color, no beauty filter, no airbrushing, slight sensor grain"
)
# 2. New faces drawn with a realism-first prompt (same shop and uniform, new seeds).
REALISM_PROMPT = (
    "candid unretouched smartphone photo of a livestream sales host in a bright modern skincare shop. A fictional "
    "Vietnamese woman in her mid twenties with an ordinary everyday face, minimal makeup, real skin texture with "
    "visible pores and slight imperfections, matte skin, dark hair tied in a neat low ponytail, relaxed friendly "
    "expression with lips closed, looking straight into the camera, wearing a plain navy blue polo shirt with thin "
    "orange trim on the collar and sleeves, no logo. She sits on a chair behind a light natural wood desk, both "
    "forearms resting on the desk. Behind her: white display shelves with warm LED strip lighting, rows of plain "
    "white and pastel skincare bottles with no labels, a few small potted plants. Mixed indoor lighting, slight "
    "sensor grain, natural colors, no text, no watermark"
)
REALISM_SEEDS = [505, 606, 707, 808]

EXTRA_SERVER_ARGS: list[str] = []  # e.g. ["--lowvram"] if the T4 runs out of memory

## 2. Môi trường và ComfyUI

In [ ]:
import json
import shutil
import subprocess
import sys
import time
import urllib.error
import urllib.request
from datetime import UTC, datetime
from pathlib import Path

import torch  # preinstalled by Kaggle/Colab with the right CUDA build

ON_KAGGLE = Path("/kaggle").exists()
ON_COLAB = "google.colab" in sys.modules
if not (ON_KAGGLE or ON_COLAB):
    raise SystemExit("Notebook này chỉ chạy trên Kaggle hoặc Colab (ADR-036), không chạy trên laptop.")
PLATFORM = "kaggle" if ON_KAGGLE else "colab"

# Weights stay out of /kaggle/working: that folder is the saved output and is capped at 20 GB.
COMFY_DIR = Path("/tmp/ComfyUI") if ON_KAGGLE else Path("/content/ComfyUI")
OUT_ROOT = (Path("/kaggle/working") if ON_KAGGLE else Path("/content")) / "avatar_out"
OUT_DIR = OUT_ROOT / CHARACTER_ID
OUT_DIR.mkdir(parents=True, exist_ok=True)


def run(*cmd: str, cwd: Path | None = None) -> str:
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd[:4])} lỗi:\n{result.stderr[-3000:]}")
    return result.stdout.strip()


if not COMFY_DIR.exists():
    run("git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI", str(COMFY_DIR))
COMFY_COMMIT = run("git", "rev-parse", "HEAD", cwd=COMFY_DIR)

# Keep the platform's CUDA build of torch: reinstalling it from PyPI can break the GPU runtime.
SKIP = {"torch", "torchvision", "torchaudio"}
reqs = []
for line in (COMFY_DIR / "requirements.txt").read_text().splitlines():
    line = line.strip()
    name = line.split(";")[0].split("[")[0]
    for sep in "<>=!~ ":
        name = name.split(sep)[0]
    if line and not line.startswith("#") and name.lower() not in SKIP:
        reqs.append(line)
run(sys.executable, "-m", "pip", "install", "-q", *reqs, "huggingface_hub")

if shutil.which("ffmpeg") is None:
    run("apt-get", "install", "-y", "-qq", "ffmpeg")

GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print(f"Nền tảng: {PLATFORM} · GPU: {GPU_NAME} · torch {torch.__version__} · ComfyUI {COMFY_COMMIT[:10]}")
if GPU_NAME == "none":
    raise SystemExit("Chưa bật GPU. Kaggle: Accelerator = GPU T4 x2. Colab: Runtime type = T4 GPU.")

## 3. Tải model (chỉ trong máy Kaggle, không về laptop)

Khoảng 17 GB.

In [ ]:
from huggingface_hub import hf_hub_download

MODELS = {
    "scene": {
        "repo": "Comfy-Org/Flux-1-Schnell",
        "file": "flux1-schnell-fp8.safetensors",
        "dir": "checkpoints",
        "license": "Apache-2.0 (FLUX.1-schnell)",
    },
}

for key, model in MODELS.items():
    started = time.monotonic()
    cached = Path(hf_hub_download(model["repo"], model["file"]))
    parts = cached.parts
    model["revision"] = parts[parts.index("snapshots") + 1] if "snapshots" in parts else "unknown"
    target = COMFY_DIR / "models" / model["dir"] / cached.name
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        target.symlink_to(cached)
    print(f"{key}: {cached.name} @ {model['revision'][:10]} ({time.monotonic() - started:.0f} s)")

## 4. Bật ComfyUI (chỉ nghe trong máy, không mở giao diện web)

In [ ]:
SERVER = "http://127.0.0.1:8188"
# Stays open while the server runs; the notebook process owns it.
server_log = open(OUT_ROOT / "comfyui.log", "w")
server = subprocess.Popen(
    [
        sys.executable,
        "main.py",
        "--listen",
        "127.0.0.1",
        "--port",
        "8188",
        "--preview-method",
        "none",
        *EXTRA_SERVER_ARGS,
    ],
    cwd=COMFY_DIR,
    stdout=server_log,
    stderr=subprocess.STDOUT,
)


def api(path: str, payload: dict | None = None) -> dict:
    data = None if payload is None else json.dumps(payload).encode()
    request = urllib.request.Request(SERVER + path, data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=30) as response:
        body = response.read()
    return json.loads(body) if body else {}


def log_tail() -> str:
    return (OUT_ROOT / "comfyui.log").read_text(errors="replace")[-3000:]


for _ in range(120):
    if server.poll() is not None:
        raise RuntimeError("ComfyUI dừng khi khởi động:\n" + log_tail())
    try:
        api("/system_stats")
        break
    except OSError:
        time.sleep(2)
else:
    raise RuntimeError("ComfyUI không lên sau 4 phút:\n" + log_tail())
print("ComfyUI đã chạy.")


def run_workflow(workflow: dict, timeout_s: float) -> dict:
    """Queue one API-format workflow and wait for its outputs."""
    try:
        prompt_id = api("/prompt", {"prompt": workflow})["prompt_id"]
    except urllib.error.HTTPError as err:
        # ComfyUI explains a rejected workflow (missing node, wrong input) in the body.
        raise RuntimeError(err.read().decode(errors="replace")[:3000]) from err
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        entry = api(f"/history/{prompt_id}").get(prompt_id)
        if entry is not None:
            if entry.get("status", {}).get("status_str") == "error":
                raise RuntimeError(json.dumps(entry["status"], ensure_ascii=False)[:3000])
            return entry["outputs"]
        if server.poll() is not None:
            raise RuntimeError("ComfyUI đã dừng giữa chừng:\n" + log_tail())
        time.sleep(5)
    raise TimeoutError(f"Quá {timeout_s:.0f} s mà workflow chưa xong")


def output_images(outputs: dict) -> list[Path]:
    files = []
    for node_output in outputs.values():
        for image in node_output.get("images", []):
            files.append(COMFY_DIR / "output" / image["subfolder"] / image["filename"])
    return files

## 5. Tạo ảnh cảnh

Bỏ ảnh trông trẻ con, giống người nổi tiếng hay người quen, méo tay hoặc méo mặt, có chữ hay logo.

In [ ]:
from IPython.display import Image, display
from PIL import Image as PILImage
from PIL import ImageStat


def scene_workflow(prompt: str, seed: int, prefix: str) -> dict:
    width, height = IMAGE_SIZE
    return {
        "1": {"class_type": "CheckpointLoaderSimple", "inputs": {"ckpt_name": MODELS["scene"]["file"]}},
        "2": {"class_type": "CLIPTextEncode", "inputs": {"text": prompt, "clip": ["1", 1]}},
        # cfg 1 ignores the negative branch; KSampler still needs one.
        "3": {"class_type": "CLIPTextEncode", "inputs": {"text": "", "clip": ["1", 1]}},
        "4": {"class_type": "EmptySD3LatentImage", "inputs": {"width": width, "height": height, "batch_size": 1}},
        "5": {
            "class_type": "KSampler",
            "inputs": {
                "model": ["1", 0],
                "positive": ["2", 0],
                "negative": ["3", 0],
                "latent_image": ["4", 0],
                "seed": seed,
                "steps": STEPS,
                "cfg": 1.0,
                "sampler_name": "euler",
                "scheduler": "simple",
                "denoise": 1.0,
            },
        },
        "6": {"class_type": "VAEDecode", "inputs": {"samples": ["5", 0], "vae": ["1", 2]}},
        "7": {"class_type": "SaveImage", "inputs": {"images": ["6", 0], "filename_prefix": prefix}},
    }


SCENES_DIR = OUT_DIR / "scenes"
SCENES_DIR.mkdir(exist_ok=True)
scenes = []
for framing in RUN_FRAMINGS:
    framing_prompt = FRAMINGS[framing]
    for seed in SEEDS:
        name = f"scene_{framing}_seed{seed}.png"
        path = SCENES_DIR / name
        started = time.monotonic()
        if not path.exists():
            prompt = f"{SCENE_PROMPT}, {framing_prompt}"
            outputs = run_workflow(scene_workflow(prompt, seed, f"{CHARACTER_ID}_scene"), timeout_s=1200)
            [image] = output_images(outputs)
            shutil.copy(image, path)
        # fp16 overflow on a T4 shows up as an all-black image.
        luma = round(ImageStat.Stat(PILImage.open(path).convert("L")).mean[0], 1)
        seconds = round(time.monotonic() - started, 1)
        scenes.append(
            {"framing": framing, "seed": seed, "file": f"scenes/{name}", "seconds": seconds, "mean_luma": luma}
        )
        print(f"{name}: {seconds} s, độ sáng trung bình {luma}")

## 5b. Làm da tự nhiên hơn và tạo mặt mới

- **Làm lại da:** phóng ảnh đã chọn lên 1,5 lần rồi lấy mẫu lại ở mức nhiễu thấp với prompt về kết cấu da. Giữ gương mặt và
  bố cục, chỉ vẽ lại chi tiết nhỏ (lỗ chân lông, lông tơ, da lì). Mức nhiễu càng cao thì da càng thật nhưng mặt càng lệch
  khỏi ảnh gốc.
- **Mặt mới:** cùng shop và đồng phục, prompt ưu tiên ảnh điện thoại chưa chỉnh da, 4 seed mới.
- `face_compare_run2.jpg` cắt vùng mặt ở độ phân giải gốc, vì ảnh thu nhỏ không thấy được khác biệt về da.

In [ ]:
def refine_workflow(source_name: str, prompt: str, denoise: float, seed: int, prefix: str) -> dict:
    return {
        "1": {"class_type": "CheckpointLoaderSimple", "inputs": {"ckpt_name": MODELS["scene"]["file"]}},
        "2": {"class_type": "CLIPTextEncode", "inputs": {"text": prompt, "clip": ["1", 1]}},
        "3": {"class_type": "CLIPTextEncode", "inputs": {"text": "", "clip": ["1", 1]}},
        "4": {"class_type": "LoadImage", "inputs": {"image": source_name}},
        "5": {
            "class_type": "ImageScaleBy",
            "inputs": {"image": ["4", 0], "upscale_method": "lanczos", "scale_by": REFINE_SCALE},
        },
        "6": {"class_type": "VAEEncode", "inputs": {"pixels": ["5", 0], "vae": ["1", 2]}},
        "7": {
            "class_type": "KSampler",
            "inputs": {
                "model": ["1", 0],
                "positive": ["2", 0],
                "negative": ["3", 0],
                "latent_image": ["6", 0],
                "seed": seed,
                "steps": STEPS,
                "cfg": 1.0,
                "sampler_name": "euler",
                "scheduler": "simple",
                "denoise": denoise,
            },
        },
        "8": {"class_type": "VAEDecode", "inputs": {"samples": ["7", 0], "vae": ["1", 2]}},
        "9": {"class_type": "SaveImage", "inputs": {"images": ["8", 0], "filename_prefix": prefix}},
    }


def record(kind: str, path: Path, started: float, **extra) -> dict:
    luma = round(ImageStat.Stat(PILImage.open(path).convert("L")).mean[0], 1)
    seconds = round(time.monotonic() - started, 1)
    print(f"{path.name}: {seconds} s, độ sáng trung bình {luma}")
    return {"kind": kind, "file": str(path.relative_to(OUT_DIR)), "seconds": seconds, "mean_luma": luma, **extra}


REFINED_DIR = OUT_DIR / "refined"
REFINED_DIR.mkdir(exist_ok=True)
refined = []
for source in REFINE_SOURCES:
    # LoadImage only reads from ComfyUI's input folder.
    (COMFY_DIR / "input").mkdir(exist_ok=True)
    shutil.copy(SCENES_DIR / source, COMFY_DIR / "input" / source)
    framing = next(f for f in FRAMINGS if f in source)
    seed = int(source.rsplit("seed", 1)[1].split(".")[0])
    prompt = f"{SCENE_PROMPT}, {FRAMINGS[framing]}, {SKIN_PROMPT}"
    for denoise in REFINE_DENOISE:
        path = REFINED_DIR / f"{Path(source).stem}_skin{round(denoise * 100):02d}.png"
        started = time.monotonic()
        if not path.exists():
            workflow = refine_workflow(source, prompt, denoise, seed, f"{CHARACTER_ID}_refine")
            [image] = output_images(run_workflow(workflow, timeout_s=1800))
            shutil.copy(image, path)
        refined.append(record("refine", path, started, source=f"scenes/{source}", denoise=denoise, seed=seed))

realism = []
for seed in REALISM_SEEDS:
    path = SCENES_DIR / f"scene_real_chest_up_seed{seed}.png"
    started = time.monotonic()
    if not path.exists():
        prompt = f"{REALISM_PROMPT}, {FRAMINGS['chest_up']}"
        [image] = output_images(run_workflow(scene_workflow(prompt, seed, f"{CHARACTER_ID}_real"), timeout_s=1200))
        shutil.copy(image, path)
    realism.append(record("realism", path, started, framing="chest_up", seed=seed))

# Contact sheet: row 1 = original + refined, row 2 = realism candidates.
THUMB = (384, 672)
rows = [
    [SCENES_DIR / s for s in REFINE_SOURCES] + [OUT_DIR / r["file"] for r in refined],
    [OUT_DIR / r["file"] for r in realism],
]
columns = max(len(row) for row in rows)
sheet = PILImage.new("RGB", (THUMB[0] * columns, THUMB[1] * len(rows)), "white")
for row_index, row in enumerate(rows):
    for column, file in enumerate(row):
        sheet.paste(PILImage.open(file).convert("RGB").resize(THUMB), (column * THUMB[0], row_index * THUMB[1]))
sheet.save(OUT_DIR / "contact_sheet_run2.jpg", quality=88)

# Face crops at full resolution, original first: skin differences are invisible in thumbnails.
FACE_BOX = (0.30, 0.25, 0.70, 0.55)  # fractions of width/height around the face in the chest_up framing
crops = []
for file in rows[0] + rows[1]:
    image = PILImage.open(file).convert("RGB")
    w, h = image.size
    box = (int(FACE_BOX[0] * w), int(FACE_BOX[1] * h), int(FACE_BOX[2] * w), int(FACE_BOX[3] * h))
    crops.append(image.crop(box).resize((420, 540)))
per_row = 4
faces = PILImage.new("RGB", (420 * per_row, 540 * -(-len(crops) // per_row)), "white")
for index, crop in enumerate(crops):
    faces.paste(crop, ((index % per_row) * 420, (index // per_row) * 540))
faces.save(OUT_DIR / "face_compare_run2.jpg", quality=92)
display(Image(filename=str(OUT_DIR / "contact_sheet_run2.jpg"), width=900))
display(Image(filename=str(OUT_DIR / "face_compare_run2.jpg"), width=900))

## 6. Hồ sơ nguồn gốc

In [ ]:
provenance = {
    "character_id": CHARACTER_ID,
    "created_by": CREATED_BY,
    "created_at": datetime.now(UTC).isoformat(timespec="seconds"),
    "kind": "ai_generated_fictional_face",
    "platform": PLATFORM,
    "platform_terms_note": "ADR-036: dùng cho giai đoạn demo; đọc lại điều khoản trước khi khách trả tiền",
    "gpu": GPU_NAME,
    "torch": torch.__version__,
    "comfyui_commit": COMFY_COMMIT,
    "models": MODELS,
    "scene": {
        "prompt": SCENE_PROMPT,
        "framings": FRAMINGS,
        "size": list(IMAGE_SIZE),
        "steps": STEPS,
        "sampler": "euler/simple, cfg 1",
        "candidates": scenes,
    },
    "skin_refine": {
        "note": "run 2, 2026-11-17: the user said the face looks plastic",
        "prompt_suffix": SKIN_PROMPT,
        "scale": REFINE_SCALE,
        "steps": STEPS,
        "outputs": refined,
    },
    "realism": {"prompt": REALISM_PROMPT, "framing": FRAMINGS["chest_up"], "outputs": realism},
}
# Run 1 wrote provenance.json; run 2 writes its own file beside it.
(OUT_DIR / "provenance_run2.json").write_text(json.dumps(provenance, ensure_ascii=False, indent=2))
server.terminate()
print("Xong:", sorted(p.name for p in OUT_DIR.rglob("*") if p.is_file()))